In [ ]:
import asyncio
import uvloop
asyncio.set_event_loop_policy(uvloop.EventLoopPolicy())

In [ ]:
vllm_image = "../hp_experiments/vllm-0191.sif"
model_name = "microsoft/phi-4"
#model_name = "Qwen/Qwen3-30B-A3B-Instruct-2507-FP8"
model_name = "Qwen/Qwen3-30B-A3B-Thinking-2507-FP8"
#model_name = "Qwen/Qwen3-235B-A22B-Instruct-2507-FP8"
model_name = "RedHatAI/Llama-3.3-70B-Instruct-FP8-dynamic"
#model_name = "meta-llama/Llama-3.1-8B-Instruct"

model_kwargs = {
    "enable_prefix_caching": True, 
    "tensor_parallel_size": 4, 
    "data_parallel_size": 1,
    "max_model_len": 2**15,
    "enable_expert_parallel": False,
}

max_requests=1024
data_dir = "data/"
port = 8000
logdir="runs/temp/"

In [ ]:
import os
import json
from tqdm import tqdm
os.makedirs(logdir, exist_ok=True)

# Load data
***

In [ ]:
def read_input_batches(fn):
    with open(fn) as f:
        lines = f.readlines()
    lines = [json.loads(x) for x in lines]
    return lines

In [ ]:
request_files = {}
for x in os.listdir(data_dir):
    if not x.endswith(".jsonl"):
        continue
    dataset_name = x.split(".")[0]
    result_dir = data_dir + dataset_name + "/" + model_name.replace("/","_") + "/batches_output.jsonl"
    if os.path.exists(result_dir):
        print(f"Skipping '{result_dir}' since it already exists")
        continue
    #if "math" in x: continue
    data = read_input_batches(data_dir + x)
    for d in data:
        d["body"]["model"] = model_name
    request_files[result_dir] = data

In [ ]:
if len(request_files) == 0:
    import sys
    print("early exiting")
    sys.exit()

# Start vllm model serving
***

In [ ]:
# build apptainer command

# some weird vllm bug...
os.environ["MPLBACKEND"] = "notebook"

cmd = 'module load apptainer && apptainer run'
cmd += ' --env "HF_HUB_OFFLINE=1"'
cmd += f' --nv --bind /mnt,/local,/user,/projects,$HOME,$WORK,$TMPDIR,$PROJECT {vllm_image} --port {port} '

mk = " ".join([f'--{k.replace("_","-")} {"" if isinstance(v, bool) else v}'.strip() for k,v in model_kwargs.items() if not isinstance(v, bool) or (isinstance(v, bool) and v != False)])
cmd += f' --model {model_name} ' + mk
cmd += ' --uvicorn-log-level error '


cmd

In [ ]:
import subprocess
import socket
import time

process = subprocess.Popen(cmd, shell=True) 

In [ ]:
def wait_for_port(host, port, process=None, timeout=60.0, poll_interval=0.25):
    """Wait until a TCP port starts accepting connections, unless the process dies."""
    deadline = time.monotonic() + timeout
    last_err = None

    while time.monotonic() < deadline:
        # Try to connect
        try:
            with socket.create_connection((host, port), timeout=1):
                print(f"Service on {host}:{port} is up!")
                return True
        except (ConnectionRefusedError, socket.timeout, OSError) as e:
            last_err = e

        # If we were given a process, exit early if it died
        if process is not None:
            rc = process.poll()
            if rc is not None:
                raise RuntimeError(f"Process exited early with code {rc}") from last_err

        time.sleep(poll_interval)

    raise TimeoutError(f"Timeout waiting for {host}:{port}")


wait_for_port("localhost", port, process=process, timeout=600)

In [ ]:
from openai import OpenAI, Timeout
timeout = Timeout(connect=1500, read=None, write=None, pool=None)
client = OpenAI(api_key="", base_url=f"http://0.0.0.0:{port}/v1/", timeout=timeout)

# Run Inference
***

In [ ]:
import concurrent

def run_request_list(data):
    use_tqdm = True
    results = [None] * len(data)

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_requests) as executor:
        future_to_index = {executor.submit(client.chat.completions.create, model=model_name, messages=item["body"]["messages"], ): i for i, item in enumerate(data)}
        for future in tqdm(
            concurrent.futures.as_completed(future_to_index),
            total=len(future_to_index),
            disable=not use_tqdm,
            desc="Completed requests",
        ):
            i = future_to_index[future]
            results[i] = future.result()

    return results

In [ ]:
results = {k:run_request_list(v) for k,v in request_files.items()}

# Write output files
***

In [ ]:
for output_file, r in results.items():
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    lines = (
        json.dumps(dict(
            id=None,
            custom_id=inp["custom_id"],
            response=dict(status_code=200, request_id=None, body=out.to_dict()),
            error=None,
        ))
        for inp, out in zip(request_files[output_file], r)
    )
    with open(output_file, "w") as f:
        f.write("\n".join(lines))

In [ ]:
import sys
sys.exit()